In [1]:
import pandas as pd
import wrds
username = "hauvi"
db = wrds.Connection(wrds_username=username)

Loading library list...
Done


data process: join tables, clean data

feature engineering: sensitivities to key rates, convexity, durations, spread duration, credit spread, expected returns, future benchmark weights, roll-down/carry (convexity, coupon) 


model formulation

# AGG Holdings from website

In [4]:
path = '../data/AGG_holdings_clean.csv'
agg_csv_df = pd.read_csv(path, skiprows=9)

# AGG Holdings from CRSP

In [5]:
agg_query = """
    SELECT * FROM crsp.fund_names WHERE ticker = 'AGG'
    """ 
agg_wrds_df = db.raw_sql(agg_query)
agg_wrds_df.sort_values(by='chgdt', ascending=False, inplace=True) 
portno = int(agg_wrds_df.sort_values("chgdt").crsp_portno.iloc[-1]) # get the most recent AGG portfolio (june 30th) idk how we can get more recent ones but whatever
col_to_keep = [ 'security_name', 'cusip', 'crsp_company_key', 'ticker', 'coupon', 'maturity_dt', 'nbr_shares', 'market_val', 'percent_tna','security_rank']
agg_crsp_df = db.raw_sql(f"""
    select {', '.join(col_to_keep)} from crsp.holdings
    where crsp_portno = {portno}
      and report_dt = '2026-06-30'
""")

The CUSIPs from AGG in CRSP are only 8 digits, but the 9th digit can be calculated by Modulus 10 Double Add Double idk google it 

In [6]:
def cusip_check_digit(cusip8):
    total = 0
    for i, ch in enumerate(cusip8.upper()):
        if ch.isdigit():
            v = int(ch)
        elif ch.isalpha():
            v = ord(ch) - ord("A") + 10
        elif ch == "*":
            v = 36
        elif ch == "@":
            v = 37
        elif ch == "#":
            v = 38
        else:
            print(f"Invalid CUSIP character: {ch}") 
        if i % 2 == 1:     
            v *= 2
        total += v // 10 + v % 10
    return str((10 - total % 10) % 10)

def to_cusip9(cusip8):
    if cusip8 is None:
        return None
    return cusip8 + cusip_check_digit(cusip8)

In [7]:
agg_crsp_df = agg_crsp_df[agg_crsp_df.cusip.notnull()]
agg_crsp_df["cusip9"] = agg_crsp_df.cusip.str.strip().str.upper().apply(to_cusip9)

Mutual CUSIP list has 12,800 of the 13,400 holdings in the CSV. 

In [8]:
csv_cusip_list = agg_csv_df.CUSIP.to_list()
crsp_cusip_list = agg_crsp_df.cusip9.to_list()
len(list(set(csv_cusip_list) & set(crsp_cusip_list)))

12807

# Treasury daily data

In [9]:
library = 'crsp'
start_date = '2020-01-01'
tfz_daily_query = f"""
    select i.tcusip         as cusip,
           d.caldt          as date,
           i.itype          as security_type,
           i.tcouprt        as coupon_rate,
           i.tdatdt         as dated_date,
           i.tmatdt         as maturity_date,
           d.tdbid          as bid_price,
           d.tdask          as ask_price,
           d.tdnomprc       as mid_price,
           d.tdnomprc_flg   as mid_price_flag,
           d.tdaccint       as accrued_interest,
           d.tdyld          as daily_yield,
           d.tdyld * 365    as annual_yield,
           d.tdduratn       as duration_days,
           d.tdduratn / 365 as duration_years,
           d.tdretnua       as daily_return,
           d.tdtotout       as total_outstanding,
           d.tdpubout       as public_outstanding
    from {library}.tfz_dly as d
    join {library}.tfz_iss as i
      on d.kycrspid = i.kycrspid
    where d.caldt >= '{start_date}'
"""

tfz_daily = db.raw_sql(tfz_daily_query, date_cols=['date', 'dated_date', 'maturity_date'])
tfz_daily.security_type = tfz_daily.security_type.astype('int')

In [ ]:
tfz_daily.head()

,cusip,date,security_type,coupon_rate,dated_date,maturity_date,bid_price,ask_price,mid_price,mid_price_flag,accrued_interest,daily_yield,annual_yield,duration_days,duration_years,daily_return,total_outstanding,public_outstanding
0,912810EE,2020-01-02,1,8.5,1990-02-15,2020-02-15,100.726562,100.75,100.738281,M,3.233696,0.000061,0.022153,44.0,0.120548,0.000219,9476.0,2843.0
1,912810EE,2020-01-03,1,8.5,1990-02-15,2020-02-15,100.679688,100.703125,100.691406,M,3.256793,0.000067,0.024609,43.0,0.117808,-0.000229,9476.0,2843.0
2,912810EE,2020-01-06,1,8.5,1990-02-15,2020-02-15,100.65625,100.679688,100.667969,M,3.326087,0.000061,0.02243,40.0,0.109589,0.000441,9476.0,2843.0
3,912810EE,2020-01-07,1,8.5,1990-02-15,2020-02-15,100.648438,100.671875,100.660156,M,3.349185,0.000059,0.02163,39.0,0.106849,0.000147,9476.0,2843.0
4,912810EE,2020-01-08,1,8.5,1990-02-15,2020-02-15,100.625,100.648438,100.636719,M,3.372283,0.000061,0.022231,38.0,0.10411,-0.000003,9476.0,2843.0


In [11]:
agg_csv_df.Sector.unique()

array(['Cash and/or Derivatives', 'Treasury', 'MBS Pass-Through',
       'Agency', 'CMBS', 'Industrial', 'Supranational',
       'Financial Institutions', 'Local Authority', 'ABS', 'Sovereign',
       'Utility'], dtype=object)

In [12]:
len(agg_csv_df[agg_csv_df.Sector.isin(['Industrial', 'Financial Institutions', 'Utility'])].CUSIP.unique())

8838

In [13]:
agg_tsy = agg_csv_df[agg_csv_df.Sector == 'Treasury'].copy()
agg_tsy['cusip8'] = agg_tsy.CUSIP.str.strip().str[:8]

agg_cusips = set(agg_tsy.cusip8)
tfz_cusips = set(tfz_daily.cusip)

missing = agg_tsy[~agg_tsy.cusip8.isin(tfz_cusips)]

print(f"AGG Treasuries: {len(agg_cusips)}")
print(f"Found in tfz_dly: {len(agg_cusips & tfz_cusips)}")
print(f"Missing: {len(missing)}")
print(f"Market value missing: {missing['Market Value'].str.replace(',', '').astype(float).sum() / agg_tsy['Market Value'].str.replace(',', '').astype(float).sum():.1%}")

missing[['Name', 'CUSIP', 'Maturity', 'Coupon (%)', 'Accrual Date', 'Market Value']]


AGG Treasuries: 297
Found in tfz_dly: 253
Missing: 44
Market value missing: 16.2%


,Name,CUSIP,Maturity,Coupon (%),Accrual Date,Market Value
3,TREASURY NOTE (OLD),91282CQQ7,"May 15, 2036",4.38,"May 15, 2026","544,523,031.40"
10,TREASURY NOTE (2OLD),91282CPZ8,"Feb 15, 2036",4.13,"Feb 15, 2026","522,872,881.66"
18,TREASURY NOTE (OLD),91282CRK9,"Aug 31, 2031",4.38,"Aug 31, 2026","452,520,882.81"
22,TREASURY NOTE,91282CQU8,"May 31, 2031",4.13,"May 31, 2026","432,304,712.13"
23,TREASURY NOTE,91282CQK0,"Apr 30, 2031",3.88,"Apr 30, 2026","431,822,367.17"
27,TREASURY NOTE,91282CQD6,"Feb 28, 2031",3.50,"Feb 28, 2026","415,156,187.91"
29,TREASURY NOTE (2OLD),91282CRA1,"Jul 31, 2031",4.38,"Jul 31, 2026","409,826,302.21"
33,TREASURY NOTE,91282CQG9,"Mar 31, 2031",3.88,"Mar 31, 2026","397,151,721.17"
42,TREASURY NOTE,91282CQX2,"Jun 30, 2031",4.13,"Jun 30, 2026","362,384,682.12"
44,TREASURY NOTE,91282CQB0,"Feb 29, 2028",3.38,"Feb 28, 2026","359,433,409.85"


# Corporate bonds daily data from Bond Dickerson

Use contrib_bond_dickerson for corporates, with a fall back to TRACE's corporate bonds

In [4]:
corporate_sectors = ['Industrial', 'Financial Institutions', 'Utility'] #from agg_csv_df
# get the corporate bond cusips 

In [5]:
db.describe_table(library=library, table='dickerson_bonds_daily')

NameError: name 'library' is not defined

In [ ]:
library = 'contrib_bond_dickerson'
table = 'dickerson_bonds_daily'
start_date = '2020-01-01'

dickerson_query = f"""
    select cusip_id              as cusip,
           permno, permco,
           trd_exctn_dt          as date,
           pr                    as clean_price,
           prfull                as full_price,
           ytm                   as yield_to_maturity,
           credit_spread,
           mod_dur               as modified_duration,
           mac_dur               as macaulay_duration,
           convexity,
           bond_maturity         as years_to_maturity,
           bond_age              as years_since_issue,
           bond_amt_outstanding  as amount_outstanding,
           prc_ew                as price_equal_wtd,
           prc_vw_par            as price_volume_wtd,
           prc_first             as price_first_trade,
           prc_last              as price_last_trade,
           prc_hi                as price_high,
           prc_lo                as price_low,
           prc_bid               as price_bid_side,
           prc_ask               as price_ask_side,
           trade_count,
           bid_count,
           ask_count,
           qvolume               as par_volume,
           dvolume               as dollar_volume,
           sp_rating, mdy_rating
    from {library}.{table}

    where trd_exctn_dt >= '{start_date}'
      and cusip_id in {corp_cusips}
""" # see how many of the cusips we get from this one data source

corp_daily_df = db.raw_sql(dickerson_query, date_cols=['date'])

we get 67% oof the corporate bond cusips from this data source

In [ ]:
corp_daily.sort_values(by='date', ascending=False)

,cusip,permno,permco,date,clean_price,full_price,yield_to_maturity,credit_spread,modified_duration,macaulay_duration,...,price_low,price_bid_side,price_ask_side,trade_count,bid_count,ask_count,par_volume,dollar_volume,sp_rating,mdy_rating
24590,98978VAV5,21936.0,21394.0,2025-03-31,104.2935,106.37791,0.049168,<NA>,6.068,6.217,...,104.128,104.25965,104.18001,16.0,5.0,6.0,1.41,1.470537,9.0,7.0
269917,03027XBD1,86111.0,34770.0,2025-03-31,65.2835,66.19624,0.057163,<NA>,14.964,15.391,...,65.033,65.15712,65.352585,9.0,3.0,5.0,5.215,3.404507,9.0,10.0
61821,89115A2E1,83835.0,29152.0,2025-03-31,96.5014,97.90009,0.050415,<NA>,5.967,6.118,...,96.343,96.515854,96.54659,37.0,10.0,12.0,10.507,10.139395,<NA>,6.0
276028,03027XBY5,86111.0,34770.0,2025-03-31,102.4133,102.642494,0.046156,<NA>,2.7,2.762,...,102.311,102.336,102.414925,11.0,1.0,5.0,12.345,12.642925,9.0,10.0
279543,03040WAD7,92614.0,52985.0,2025-03-31,110.5418,113.60026,0.054228,<NA>,8.459,8.689,...,110.341,<NA>,110.72057,5.0,<NA>,3.0,0.389,0.430006,6.0,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
457348,29250NAZ8,76932.0,2381.0,2020-01-02,102.035,102.477715,0.028861,0.010009,8.418,8.54,...,102.035,<NA>,102.035,1.0,<NA>,1.0,0.5,0.510175,8.0,9.0
264187,03027XAW0,86111.0,34770.0,2020-01-02,107.2527,108.74104,0.029282,0.010523,7.972,8.089,...,107.083,107.20361,107.51834,14.0,4.0,6.0,14.045,15.063615,10.0,10.0
480297,595620AN5,11170.0,9020.0,2020-01-02,119.6198,120.60974,0.032418,0.008986,15.813,16.069,...,119.596,<NA>,119.61975,2.0,<NA>,2.0,0.4,0.478479,5.0,3.0
181036,02209SBD4,13901.0,21398.0,2020-01-02,111.4809,113.3742,0.033277,0.0147,7.337,7.459,...,111.446,111.452866,111.5584,10.0,4.0,3.0,6.834,7.618597,9.0,7.0


In [ ]:
corp_daily_df.cusip.nunique() /len(corp_cusips)
remaining_corp_cusips = list(set(corp_cusips) - set(corp_daily_df.cusip)) # make this our new query set

0.6748133061778683

# WRDS' Bond Returns dataset records the last traded price of the month for given corporate bonds
Really, they just do data processing on other existing data libraries in their catalogue
can read about here https://wrds-www.wharton.upenn.edu/documents/248/WRDS_Corporate_Bond_Database_Manual.pdf

In [ ]:
db.describe_table(library='wrdsapps_bondret', table='bondret')

Approximately 4590326 rows in wrdsapps_bondret.bondret.


,name,nullable,type,comment
0,date,True,DATE,Monthly Date
1,issue_id,True,DOUBLE PRECISION,Mergent FISD ISSUE ID
2,cusip,True,VARCHAR(9),CUSIP
3,bond_sym_id,True,VARCHAR(15),TRACE Bond Symbol
4,bsym,True,DOUBLE PRECISION,Bloomberg Identifier
5,isin,True,VARCHAR(12),ISIN
6,company_symbol,True,VARCHAR(6),Company Symbol (issuer stock ticker)
7,bond_type,True,VARCHAR(4),Corporate Bond Types
8,security_level,True,VARCHAR(4),"Security Level: secured, senior or subordinated"
9,conv,True,DOUBLE PRECISION,Flag Convertible


In [ ]:
corporate_sectors = ['Industrial', 'Financial Institutions', 'Utility'] #from agg_csv_df
corp_cusips = tuple(agg_csv_df.loc[agg_csv_df.Sector.isin(corporate_sectors), 'CUSIP'].str.strip().unique())
library = 'wrdsapps_bondret'
table = 'bondret'
start_date = '2020-01-01'

bondret_query = f"""
    select cusip,
           isin,
           date                 as month_end,
           bond_type,
           security_level,
           conv                 as convertible,
           offering_date,
           offering_amt         as offering_amount,
           dated_date,
           maturity             as maturity_date,
           tmt                  as years_to_maturity,
           coupon               as coupon_rate,
           coupamt              as coupon_amount,
           coupacc              as accrued_coupon,
           nextcoup             as next_coupon_date,
           amount_outstanding,
           r_sp                 as sp_rating,
           r_mr                 as moodys_rating,
           r_fr                 as fitch_rating,
           rating_num,
           rating_cat,
           rating_class,
           price_eom            as price_end_of_month,
           price_ldm            as price_last_day_of_month,
           price_l5m            as price_last_5_days_of_month,
           price_eom_flg        as price_flag,
           ret_eom              as return_end_of_month,
           ret_ldm              as return_last_day_of_month,
           ret_l5m              as return_last_5_days_of_month,
           yield                as yield_to_maturity,
           t_spread             as spread_over_treasury,
           duration,
           t_date               as last_trade_date,
           t_volume             as last_trade_par_volume,
           t_dvolume            as last_trade_dollar_volume,
           t_yld_pt             as last_trade_yield,
           gap                  as months_since_last_price,
           defaulted
    from {library}.{table}
    where date >= '{start_date}'
      and cusip in {corp_cusips}
"""

bondret = db.raw_sql(bondret_query, date_cols=['month_end', 'offering_date', 'dated_date',
                                              'maturity_date', 'next_coupon_date', 'last_trade_date'])


this has 76% of corporate cusips from AGG, on a monthly freq (because corps dont trade that frequently so they just take the monthly end price)

In [ ]:
len(bondret.cusip.unique())

6690

In [ ]:
bondret.cusip.nunique() / len(corp_cusips)

0.756958587915818

# TRACE Enhanced Dataset for Agency Bonds 

In [ ]:
library = 'trace'
table = 'trace_agency'
start_date = '2020-01-01'

# Municipal bonds from MSRB (municipal securities rulemaking board)

problem is this isnt daily data its by trades so u gotta aggregate or else the rows will be fucked


In [ ]:
library = 'msrb'
table = 'msrb'
db.get_table(library=library, table=table, obs=10)

,rtrs_control_number,trade_type_indicator,cusip,security_description,dated_date,coupon,maturity_date,when_issued_indicator,assumed_settlement_date,trade_date,...,brokers_broker_indicator,weighted_price_indicator,offer_price_takedown_indicator,rtrs_publish_date,rtrs_publish_time,version_number,uv_dollar_price_indicator,ats_indicator,ntbc_indicator,cusip6
0,C2138C5B651F2141,P,000369BZ0,ABAG FIN AUTH FOR NONPROFIT COMB FING 3,1993-11-01,5.6,2023-11-01,<NA>,<NA>,2005-01-12,...,<NA>,<NA>,<NA>,2005-02-10,00:54:37,1.0,<NA>,<NA>,<NA>,000369
1,66573B13656D887A,S,000369BZ0,ABAG FIN AUTH FOR NONPROFIT COMB FING 3,1993-11-01,5.6,2023-11-01,<NA>,<NA>,2005-01-12,...,<NA>,<NA>,<NA>,2005-02-10,00:54:40,1.0,<NA>,<NA>,<NA>,000369
2,8175FF6CB09D4070,S,000369BZ0,ABAG FIN AUTH FOR NONPROFIT COMB FING 3,1993-11-01,5.6,2023-11-01,<NA>,<NA>,2005-01-19,...,<NA>,<NA>,<NA>,2005-02-17,00:30:22,1.0,<NA>,<NA>,<NA>,000369
3,EC67643C274169E5,P,000369BZ0,ABAG FIN AUTH FOR NONPROFIT COMB FING 3,1993-11-01,5.6,2023-11-01,<NA>,<NA>,2005-04-28,...,<NA>,<NA>,<NA>,2005-05-27,00:39:28,1.0,<NA>,<NA>,<NA>,000369
4,33CFFF435742005A,S,000369BZ0,ABAG FIN AUTH FOR NONPROFIT COMB FING 3,1993-11-01,5.6,2023-11-01,<NA>,<NA>,2005-05-02,...,<NA>,<NA>,<NA>,2005-05-28,00:34:06,1.0,<NA>,<NA>,<NA>,000369
5,151CFC7A0B0D712A,D,000369BZ0,ABAG FIN AUTH FOR NONPROFIT COMB FING 3,1993-11-01,5.6,2023-11-01,<NA>,2005-07-15,2005-07-12,...,S,<NA>,<NA>,2005-08-10,00:39:18,1.0,<NA>,<NA>,<NA>,000369
6,01257CEF90D6E708,D,000369BZ0,ABAG FIN AUTH FOR NONPROFIT COMB FING 3,1993-11-01,5.6,2023-11-01,<NA>,2005-07-15,2005-07-12,...,P,<NA>,<NA>,2005-08-10,00:39:19,1.0,<NA>,<NA>,<NA>,000369
7,D8A8609D439DD481,D,000369BZ0,ABAG FIN AUTH FOR NONPROFIT COMB FING 3,1993-11-01,5.6,2023-11-01,<NA>,2005-07-15,2005-07-12,...,<NA>,<NA>,<NA>,2005-08-10,00:39:19,1.0,<NA>,<NA>,<NA>,000369
8,14F7E4664EBA6598,P,000369BZ0,ABAG FIN AUTH FOR NONPROFIT COMB FING 3,1993-11-01,5.6,2023-11-01,<NA>,<NA>,2005-07-12,...,<NA>,<NA>,<NA>,2005-08-10,00:39:18,1.0,<NA>,<NA>,<NA>,000369
9,692E9983C64DA72C,S,000369BZ0,ABAG FIN AUTH FOR NONPROFIT COMB FING 3,1993-11-01,5.6,2023-11-01,<NA>,<NA>,2005-07-13,...,<NA>,<NA>,<NA>,2005-08-11,00:39:31,1.0,<NA>,<NA>,<NA>,000369


In [ ]:
start_date = '2020-01-01'
msrb_query = f"""


"""

# MBS pass through data

In [ ]:
mbs_cusips = agg_csv_df.loc[agg_csv_df.Sector == 'MBS Pass-Through', 'CUSIP'].str.strip().unique().tolist()
start_date = '2020-01-01'

def mbs_query(cusips):
    cusip_list = ",".join(f"'{c}'" for c in cusips)
    return f"""
        select cusip_id                                      as cusip,
               trd_exctn_dt                                  as date,
               sum(rptd_pr * entrd_vol_qt) / sum(entrd_vol_qt) as price_volume_wtd,
               min(rptd_pr)                                  as price_low,
               max(rptd_pr)                                  as price_high,
               sum(entrd_vol_qt)                             as volume,
               count(*)                                      as trade_count,
               max(factor)                                   as factor
        from trace_enhanced.trace_spds_mbs_enhanced
        where trd_exctn_dt >= '{start_date}'
          and cusip_id in ({cusip_list})
          and trc_st = 'T'
          and rptd_pr > 0
          and entrd_vol_qt > 0
        group by cusip_id, trd_exctn_dt
    """

mbs_daily = pd.concat(
    [db.raw_sql(mbs_query(mbs_cusips[i:i+1000]), date_cols=['date'])
     for i in range(0, len(mbs_cusips), 1000)],
    ignore_index=True,
)

In [ ]:
mbs_daily.sort_values(by='date', ascending=False)

,cusip,date,price_volume_wtd,price_low,price_high,volume,trade_count,factor
207253,36179TNR2,2025-03-17,91.996093,91.996093,91.996093,9830000.0,1,0.0
19000,36179XLH7,2025-03-17,98.71875,98.71875,98.71875,2000000.0,1,0.0
41932,3132DWGH0,2025-03-17,100.031109,99.5156,100.03125,22006000.0,5,0.0
101463,3618N5A33,2025-03-17,98.63954,98.21875,100.192969,1710000.0,15,0.0
42528,36180AAE3,2025-03-17,98.054687,98.054687,98.054687,4500000.0,1,0.0
...,...,...,...,...,...,...,...,...
172521,3140Q7EF0,2020-01-02,105.328125,105.328125,105.328125,350000.0,1,0.70304
172520,3140Q75P8,2020-01-02,103.625,103.625,103.625,60000.0,3,0.838709
172519,3140J5FG9,2020-01-02,104.15984,104.0625,104.171875,61800.0,3,0.0
89339,3128MJZB9,2020-01-02,101.973967,101.02359,102.02,431000.0,19,0.0


In [ ]:
mbs_daily.cusip.unique()

<StringArray>
['31418D3N0', '3132DWMX8', '36179W7K8', '36179WLP1', '36179V4U1', '3128MJZB9',
 '36179WNF1', '3132D6AU4', '3133KPUC0', '3132DWDY6',
 ...
 '3140H3NK8', '3140H5XU0', '3132J62B9', '3140F2GY0', '31418BUC8', '312941UZ8',
 '3132GRB75', '3132D6JE1', '3131XXLN9', '3138YUA68']
Length: 2389, dtype: string

In [ ]:
agg_csv_df.Sector.unique()

array(['Cash and/or Derivatives', 'Treasury', 'MBS Pass-Through',
       'Agency', 'CMBS', 'Industrial', 'Supranational',
       'Financial Institutions', 'Local Authority', 'ABS', 'Sovereign',
       'Utility'], dtype=object)

In [ ]:
agg_csv_df[agg_csv_df.Sector == 'MBS Pass-Through'].CUSIP.nunique()

2934

In [ ]:
2389/2934

0.8142467620995228

# missing: 

mbs pass throughs (21% of agg)

33% of corps (33% of 25%)

treasury curve for key-rate sensitivity data 

cmbs, abs, sovereign, supranational (3% of agg)